# 🧠 ChromaDB RAG Vectorstore — Legal & Medical PDF Ingestion

**Part of:** SafeVisionAI · IIT Madras Road Safety Hackathon 2026
**Output:** `chroma_db/` directory → deployed to `chatbot_service/data/chroma_db/`

This notebook builds the **Retrieval-Augmented Generation (RAG)** knowledge base for the SafeVisionAI chatbot.
It ingests Indian legal documents (Motor Vehicles Act, MoRTH circulars) and first-aid medical PDFs,
chunks them, embeds them using `sentence-transformers`, and stores them in a **ChromaDB** vector store.

---
### 🗂️ Source Documents
| Category | Files | Source |
|----------|-------|--------|
| Legal | Motor Vehicles Act 2019, MoRTH 2022 | `download_legal_pdfs.py` |
| Medical | First Aid guides, Emergency protocols | `download_legal_pdfs.py` |

### 🔄 Pipeline
```
PDFs → pdfplumber chunks → sentence-transformer embeddings → ChromaDB index
```

> 💡 The resulting `chroma_db/` is what the chatbot queries at runtime for grounded answers.

## 🔧 Step 1 — Install Dependencies

Installs the full RAG stack:
- `chromadb` — local vector database for semantic search
- `sentence-transformers` — `all-MiniLM-L6-v2` model for text embeddings
- `pdfplumber` — PDF text extraction with page layout awareness
- `langchain` — document chunking utilities

In [ ]:
import time
from IPython.display import Javascript
display(Javascript('''
  function ClickConnect(){
    document.querySelector('#top-toolbar > colab-connect-button').click()
  }
  setInterval(ClickConnect, 60000)
'''))
print('Anti-disconnect activated')


<IPython.core.display.Javascript object>

Anti-disconnect activated


## 📂 Step 2 — Upload PDF Documents

Upload all legal and medical PDFs from:
```
chatbot_service/data/legal/
chatbot_service/data/medical/
```

> 📄 Expected PDFs: Motor_Vehicles_Act.pdf, MoRTH_2022_Report.pdf, first_aid_guide.pdf, etc.

In [ ]:
# Cell 1 — Install RAG tools
!pip install chromadb==0.5.3 sentence-transformers pypdf pdfplumber langchain langchain-community pandas "numpy<2.0.0" -q
print("✅ Installations complete")


✅ Installations complete


## ✂️ Step 3 — Extract & Chunk PDF Text

Uses `pdfplumber` to extract text from each PDF page,
then splits into fixed-size chunks (512 tokens) with 50-token overlap.

Chunking ensures the embedding model sees coherent, context-rich passages
rather than arbitrarily cut sentences.

In [ ]:
# Cell 2 — Initialize Embedder Models
# We use TWO embedders: English for legal text, Multilingual for standard files
from sentence_transformers import SentenceTransformer

print("Loading English embedder... (this takes a moment)")
en_embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("Loading Multilingual embedder... (this takes a moment)")
multi_embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("✅ Embedders ready")


Loading English embedder... (this takes a moment)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedders ready


## 🔢 Step 4 — Generate Embeddings

Uses the `all-MiniLM-L6-v2` sentence-transformer model to convert each text chunk
into a 384-dimensional embedding vector.

| Model | Dimensions | Speed | Quality |
|-------|-----------|-------|---------|
| all-MiniLM-L6-v2 | 384 | Fast | Good for semantic QA |

In [ ]:
# Cell 3 — Initialize Chroma Collections
import chromadb
client = chromadb.PersistentClient(path='/content/chroma_db')
legal_col = client.get_or_create_collection(name='legal_docs', metadata={'hnsw:space': 'cosine'})
general_col = client.get_or_create_collection(name='general_docs', metadata={'hnsw:space': 'cosine'})
print("✅ Vector DB collections created")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector DB collections created


## 💾 Step 5 — Build & Persist ChromaDB Index

Creates a persistent ChromaDB collection and upserts all embedded chunks.
The resulting `chroma_db/` folder contains the SQLite + vector index files.

> 📦 Output size: ~50-100MB depending on number of PDFs ingested.

In [ ]:
# Cell 3.5 — Create Folders and Upload Files
import os, shutil
from google.colab import files

os.makedirs('/content/rag_data/legal', exist_ok=True)
os.makedirs('/content/rag_data/medical', exist_ok=True)
os.makedirs('/content/rag_data/accidents', exist_ok=True)

print("▶ UPLOAD YOUR FILES NOW (Select your PDFs and CSVs from your PC):")
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.pdf'):
        if 'trauma' in fname or 'who' in fname or 'A72' in fname:
            shutil.move(fname, f'/content/rag_data/medical/{fname}')
        else:
            shutil.move(fname, f'/content/rag_data/legal/{fname}')
    elif 'accident' in fname or 'adsi' in fname or 'kaggle' in fname:
        shutil.move(fname, f'/content/rag_data/accidents/{fname}')
    else:
        shutil.move(fname, f'/content/rag_data/{fname}')

print("✅ Files uploaded and organized successfully!")


▶ UPLOAD YOUR FILES NOW (Select your PDFs and CSVs from your PC):


Saving violations_seed.csv to violations_seed.csv
✅ Files uploaded and organized successfully!


## 📥 Step 6 — Download ChromaDB

Zips the `chroma_db/` directory and downloads it for deployment.
Place the extracted folder at: `chatbot_service/data/chroma_db/`

The chatbot service auto-loads this at startup — no rebuild needed.

In [ ]:
# Cell 4 — Helper functions for Document Chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd, json, glob, os

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def add_pdf(path, collection, embedder, tag):
    loader = PyPDFLoader(path)
    chunks = splitter.split_documents(loader.load())
    texts = [c.page_content for c in chunks]
    embeds = embedder.encode(texts).tolist()
    collection.add(
        documents=texts, embeddings=embeds,
        ids=[f'{tag}_{i}' for i in range(len(texts))],
        metadatas=[{'source': tag}] * len(texts)
    )
    print(f"📄 Generated {len(texts)} chunks for {tag}")


In [ ]:
# Cell 5 — Parse & Embed strictly Legal PDFs
for pdf in glob.glob('/content/rag_data/legal/*.pdf'):
    add_pdf(pdf, legal_col, en_embedder, os.path.basename(pdf))
print("✅ Legal PDFs encoded successfully into legal_col")


✅ Legal PDFs encoded successfully into legal_col


In [ ]:
# Cell 6 — Parse & Embed Multilingual CSV/Rules
if os.path.exists('/content/rag_data/violations_seed.csv'):
    print("Violations file found, embedding into general_col...")
    viol = pd.read_csv('/content/rag_data/violations_seed.csv')
    viol_texts = [f"Section {r.section}: {r.get('description_en', '')}" for _,r in viol.iterrows()]
    general_col.add(
        documents=viol_texts, embeddings=multi_embedder.encode(viol_texts).tolist(),
        ids=[f'viol_{i}' for i in range(len(viol_texts))]
    )
    print(f"✅ Embedded {len(viol_texts)} rule rows.")
else:
    print("⚠️ violations_seed.csv not found, skipping...")


Violations file found, embedding into general_col...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ Embedded 28 rule rows.


In [ ]:
# Cell 7 — Download Database zip
import shutil
from google.colab import files

print("Zipping Vectorstore...")
shutil.make_archive('/content/chroma_db_export', 'zip', '/content/chroma_db')
files.download('/content/chroma_db_export.zip')


Zipping Vectorstore...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("🔍 Testing Vector Database Retrieval...")

# Test 1: Ask the violations (general_col) about helmets
print("\n--- Testing Violations Database ---")
viol_results = general_col.query(
    query_texts=["What is the fine for driving without a helmet?"],
    n_results=2
)
for doc in viol_results['documents'][0]:
    print("Found rule:", doc)

# Test 2: Ask the Legal PDFs (legal_col)
print("\n--- Testing Legal PDFs Database ---")
pdf_results = legal_col.query(
    query_texts=["helmet penalty section 194D"],
    n_results=2
)
for doc in pdf_results['documents'][0]:
    print("Found chunk:", doc[:150], "...")


🔍 Testing Vector Database Retrieval...

--- Testing Violations Database ---


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 25.2MiB/s]
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Found rule: Section Section 194D: Failure to wear protective headgear on a two-wheeler
Found rule: Section Sections 3/181: Driving without a valid driving licence

--- Testing Legal PDFs Database ---
